# BTS AOS testing BLOCK

This test takes a set of exposures with different AOS pipelines to test pipeline integration.

Created on: 2026-05-12

Author: Chris Suberlak

In [1]:
from lsst.ts.observing import ObservingBlock, ObservingScript 
from lsst.ts.aos.analysis import build_configuration_schema
import os

In [23]:
current_path = os.getcwd()
block_number = 'T729'
program = "BLOCK-T729"
constraints = []
properties = {}

In [24]:
configuration_schema = build_configuration_schema(block_number, properties)
print(configuration_schema)

$schema: http://json-schema.org/draft-07/schema#
title: BLOCK-T729 configuration
description: Configuration for BLOCK-T729.
type: object
properties:



In [25]:
pipelines = ["DANISH","TIE",  "AI_DONUT",  "UNPAIRED_DANISH", "TARTS_UNPAIRED", "WCS_DANISH_BIN_1",
             "WCS_DANISH_BIN_2"
            ]
scripts = []

for pipeline in pipelines:
    
     # 1. Set the AOS pipeline via OCPS
    set_pipeline_script = ObservingScript(
        name="run_command.py",
        standard=True,
        parameters=dict(
            component="OCPS",
            cmd="execute",                      
            parameters=dict(pipeline=pipeline),  
 
        ),
    )
    scripts.append(set_pipeline_script)

    # 2. Take an exposure 
    take_image_script = ObservingScript(
        name="maintel/take_image_lsstcam.py",
        standard=True,
        parameters= dict(
            program="$program",
            reason=f"AOS_{pipeline}",
            exp_times=30,
            image_type="ACQ",
            nimages=1,
        )
    )

    scripts.append(take_image_script)

In [26]:
block = ObservingBlock(
    name = program,
    program = program,
    configuration_schema=configuration_schema,
    scripts = scripts,
)

### Save configurable block

In [27]:
current_path

'/sdf/data/rubin/user/scichris/WORK/aos_packages/ts_aos_analysis/notebooks/json_blocks'

In [28]:
block.model_dump_json(indent=2)

#output_file_path = f'{current_path}/aos/ts_config_ocs/Scheduler/observing_blocks_maintel/AOS/LUTs/{program}.json'
output_file_path = f'{current_path}/{program}.json'
with open(output_file_path, 'w') as file:
    file.write(block.model_dump_json(indent=2))